## Cell 1: Google Colab Setup & Google Drive Mount

In [ ]:
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive mounted successfully")
else:
    print("Running locally (not in Google Colab)")

# Set the path to your CNRS project in Google Drive
# Example: '/content/drive/My Drive/CNRS'
if IN_COLAB:
    GDRIVE_PROJECT_PATH = '/content/drive/MyDrive/audio_simon_moutier'
else:
    GDRIVE_PROJECT_PATH = 'C:/Users/Simon/Desktop/CNRS'

print(f"Project path: {GDRIVE_PROJECT_PATH}")

Mounted at /content/drive
✓ Google Drive mounted successfully
Project path: /content/drive/MyDrive/audio_simon_moutier


## Cell 2: Install Dependencies

In [ ]:
if IN_COLAB:
    print("Installing required packages...")
    !pip install -q pyreadr scikit-learn joblib pandas numpy
    print("All packages installed")

Installing required packages...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 788.3/788.3 kB 12.5 MB/s eta 0:00:00
✓ All packages installed


## Cell 3: Imports

In [ ]:
import os
import re
import time
import joblib
import numpy as np
import pandas as pd

from datetime import datetime
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold
from sklearn.decomposition import PCA
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    balanced_accuracy_score,
    f1_score
)

print("All imports successful")

✓ All imports successful


## Cell 4: Configuration & Paths

In [ ]:
############################################
# PATHS
############################################

path_to_export = os.path.join(GDRIVE_PROJECT_PATH, "Results/Random_Forest")
path_to_embeddings = os.path.join(GDRIVE_PROJECT_PATH, "embeddings/dinov2_embeddings_subset.csv")

print(f"Export path: {path_to_export}")
print(f"Embeddings path: {path_to_embeddings}")

# Verify paths exist
if not os.path.exists(path_to_embeddings):
    print(f"WARNING: Embeddings file not found at {path_to_embeddings}")
if not os.path.exists(path_to_export):
    print(f"WARNING: Export directory not found at {path_to_export}")

Export path: /content/drive/MyDrive/audio_simon_moutier/Results/Random_Forest
Embeddings path: /content/drive/MyDrive/audio_simon_moutier/embeddings/dinov2_embeddings_subset.csv


## Cell 5: Options & Parameters

In [5]:
############################################
# OPTIONS
############################################

use_label_group_modified = True
group_noise = True
downsampling = False
use_custom_weights = True
run_tuning = False
ungrouped_labels = False

# OPTIMIZATION OPTIONS
use_pca_reduction = False  # Réduire 1024 → ~80 features
n_pca_components = 0.95
reduce_cv_splits = False  # 5 → 3 splits pour économiser du temps

print("Configuration:")
print(f"  - Use label group modified: {use_label_group_modified}")
print(f"  - Group noise: {group_noise}")
print(f"  - Downsampling: {downsampling}")
print(f"  - Custom weights: {use_custom_weights}")
print(f"  - PCA reduction: {use_pca_reduction} ({n_pca_components} components)")
print(f"  - Reduce CV splits: {reduce_cv_splits}")

Configuration:
  - Use label group modified: True
  - Group noise: True
  - Downsampling: False
  - Custom weights: True
  - PCA reduction: False (0.95 components)
  - Reduce CV splits: False


## Cell 6: Create Output Folder

In [6]:
############################################
# START
############################################

start_time = time.time()

print("\n=====================================================")
print(" RANDOM FOREST TRAINING")
print("=====================================================")
print()

############################################
# 1. CREATE OUTPUT FOLDER
############################################

print("[1/11] Creating output folder...")

existing_rf = [
    d for d in os.listdir(path_to_export)
    if re.match(r"RF_\d+", d)
]

rf_numbers = [int(d.split("_")[1]) for d in existing_rf]

next_x = 1 if len(rf_numbers) == 0 else max(rf_numbers) + 1

path_rf_output = os.path.join(path_to_export, f"RF_{next_x}")

os.makedirs(path_rf_output, exist_ok=True)

print(f"Output folder: {path_rf_output}\n")


 RANDOM FOREST TRAINING

[1/11] Creating output folder...
Output folder: /content/drive/MyDrive/audio_simon_moutier/Results/Random_Forest/RF_3



## Cell 7: Load Data

In [7]:
############################################
# 2. LOAD DATA
############################################

print("[2/11] Loading embeddings...")

df = pd.read_csv(path_to_embeddings)

print(f"Loaded dataframe shape: {df.shape}")

[2/11] Loading embeddings...
Loaded dataframe shape: (10325, 1025)


## Cell 8: Extract Metadata

In [8]:
############################################
# 3. EXTRACT METADATA
############################################

print("[3/11] Extracting metadata...")

# Extract label
df["label"] = df["filename"].str.extract(r"(.*)(?=_HiP)")
df["label"] = df["label"].astype("category")

# Extract id
df["id"] = df["filename"].str.extract(r"(HiP[^_]+)")

# Species
df["specie"] = np.where(
    df["id"].isin(["HiPsh441", "HiPsh435"]),
    "hyaena",
    np.where(
        df["id"].isin(["HiP616", "HiP320", "HiP633"]),
        "lion",
        "unknown"
    )
)

print("Metadata extraction done.\n")

[3/11] Extracting metadata...
Metadata extraction done.



## Cell 9: Filter Rare Classes

In [9]:
############################################
# 4. FILTER RARE CLASSES
############################################

print("[4/11] Filtering rare classes...")

seuil = 30

exceptions = [
    "roar_period",
    "wildebeest_scream",
    "lion_roar",
    "buffalo",
    "roar_o_period",
    "lion_roar_period",
    "whoop_o_period",
    "scream_prey",
    "growl_thr",
    "growl_thr_o",
    "alarm_call_o",
    "jap_o",
    "rumble_o",
    "warthog",
    "whine",
    "scream"
]

counts = df["label"].value_counts()

valid_labels = counts[counts >= seuil].index.tolist() + exceptions

initial_n = len(df)

df = df[df["label"].isin(valid_labels)].copy()

final_n = len(df)

print(f"Samples before filtering: {initial_n}")
print(f"Samples after filtering : {final_n}")
print(f"Remaining labels         : {df['label'].nunique()}\n")

[4/11] Filtering rare classes...
Samples before filtering: 10325
Samples after filtering : 9946
Remaining labels         : 37



## Cell 10: Group Labels

In [10]:
############################################
# 6. GROUP LABELS
############################################

print("[6/11] Grouping labels...")

csv_path = os.path.join(path_to_export, "grouped_labels_V2.csv")

if not ungrouped_labels:

    labels_df = pd.read_csv(csv_path)

    group_map = {
        1: "background",
        2: "crunch",
        3: "roar",
        4: "whoop",
        5: "prey_scream",
        6: "h_noise",
        7: "l_noise",
        8: "whoop_o",
        9: "roar_o"
    }

    labels_df["group_name"] = (
        labels_df["group"]
        .map(group_map)
        .fillna("unclassified")
    )

    if group_noise:

        labels_df["group_name"] = labels_df["group_name"].replace({
            "h_noise": "noise",
            "l_noise": "noise"
        })

    df = df.merge(
        labels_df[["label", "group_name"]],
        on="label",
        how="left"
    )

else:

    df["group_name"] = df["label"]

print("Grouped labels distribution:\n")
print(df["group_name"].value_counts())
print()

[6/11] Grouping labels...
Grouped labels distribution:

group_name
noise         3865
background    3623
crunch        1827
roar_o         273
roar           182
whoop_o        131
whoop           45
Name: count, dtype: int64



## Cell 11: Prepare Features & Weights

In [11]:
############################################
# 8. FEATURES
############################################

print("[8/11] Preparing features...")

feature_cols = [
    c for c in df.columns
    if c.startswith("X") or c.startswith("dim")
]

X = df[feature_cols]
y = df["group_name"]

print(f"Number of features (original): {len(feature_cols)}")
print(f"Feature matrix shape: {X.shape}")

# PCA REDUCTION (optional)
if use_pca_reduction:
    print(f"\nApplying PCA reduction to {n_pca_components} components...")
    pca = PCA(n_components=n_pca_components, random_state=42)
    X = pd.DataFrame(
        pca.fit_transform(X),
        columns=[f"PC_{i+1}" for i in range(n_pca_components)]
    )
    print(f"Explained variance ratio: {pca.explained_variance_ratio_.sum():.4f}")
    print(f"Features after PCA: {X.shape[1]}")

print(f"\nFinal feature matrix shape: {X.shape}\n")

############################################
# 9. WEIGHTS
############################################

print("[9/11] Computing sample weights...")

class_counts = y.value_counts()

if not use_custom_weights:

    weights = y.map(
        lambda x: 1 / class_counts[x]
    )

else:

    priorite = {
        "background": 1,
        "noise": 1,
        "crunch": 1,
        "roar": 1.5,
        "roar_o": 1.5,
        "whoop": 2,
        "whoop_o": 2
    }

    weights = y.map(
        lambda x: (
            1 / class_counts[x]
        ) * priorite.get(x, 1)
    )

weights = weights / weights.mean()

print("Weights computed.\n")

[8/11] Preparing features...
Number of features (original): 1024
Feature matrix shape: (9946, 1024)

Final feature matrix shape: (9946, 1024)

[9/11] Computing sample weights...
Weights computed.



## Cell 12: Setup GroupKFold & Random Forest

In [ ]:
############################################
# 10. GROUP K-FOLD
############################################

print("[10/11] Preparing GroupKFold...")

df["original_file_id"] = (
    df["filename"]
    .str.extract(r"(HiP.+?)(?=_idx)")
)

groups = df["original_file_id"]

cv_splits = 3 if reduce_cv_splits else 5

gkf = GroupKFold(n_splits=cv_splits)

print(f"GroupKFold ready with {cv_splits} splits.\n")

############################################
# 11. MODEL
############################################

print("[11/11] Initializing Random Forest...")

rf = RandomForestClassifier(
    n_estimators=200,
    max_features="sqrt",
    criterion="gini",
    min_samples_leaf=5,
    n_jobs=-1,
    class_weight=None,
    verbose=1,
    random_state=42
)

print(f"Model initialized (matching train_audio_model.R):")
print(f"  - n_estimators: 500")
print(f"  - max_features: 128")
print(f"  - criterion: gini")
print(f"  - min_samples_leaf: 5\n")

[10/11] Preparing GroupKFold...
GroupKFold ready with 5 splits.

[11/11] Initializing Random Forest...
Model initialized (matching train_audio_model.R):
  - n_estimators: 500
  - max_features: 128
  - criterion: gini
  - min_samples_leaf: 5



## Cell 13: Cross-Validation Training

In [13]:
############################################
# TRAINING WITH CV
############################################

print("=====================================================")
print(" CROSS VALIDATION")
print("=====================================================")
print()

y_pred = np.empty(len(y), dtype=object)

fold = 1

cv_start = time.time()

for train_idx, test_idx in gkf.split(X, y, groups):

    fold_start = time.time()

    print(f"Fold {fold}/5")
    print("-" * 40)

    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    y_train = y.iloc[train_idx]

    w_train = weights.iloc[train_idx]

    print(f"Train samples: {len(train_idx)}")
    print(f"Test samples : {len(test_idx)}")

    print("Training model...")

    rf.fit(
        X_train,
        y_train,
        sample_weight=w_train
    )

    print("Predicting...")

    y_pred[test_idx] = rf.predict(X_test)

    fold_f1 = f1_score(
        y.iloc[test_idx],
        y_pred[test_idx],
        average="macro"
    )

    elapsed_fold = time.time() - fold_start

    print(f"Fold Macro F1: {fold_f1:.4f}")
    print(f"Fold duration: {elapsed_fold:.2f} sec\n")

    fold += 1

cv_elapsed = time.time() - cv_start

print("Cross-validation completed.")
print(f"Total CV time: {cv_elapsed/60:.2f} minutes\n")

 CROSS VALIDATION

Fold 1/5
----------------------------------------
Train samples: 7956
Test samples : 1990
Training model...


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=-1)]: Done  46 tasks      | elapsed:    9.8s
[Parallel(n_jobs=-1)]: Done 196 tasks      | elapsed:   44.4s
[Parallel(n_jobs=-1)]: Done 200 out of 200 | elapsed:   45.1s finished
[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  46 tasks      | elapsed:    0.0s
[Parallel(n_jobs=2)]: Done 196 tasks      | elapsed:    0.1s
[Parallel(n_jobs=2)]: Done 200 out of 200 | elapsed:    0.1s finished


Predicting...
Fold Macro F1: 0.7096
Fold duration: 45.35 sec

Fold 2/5
----------------------------------------
Train samples: 7957
Test samples : 1989
Training model...


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=-1)]: Done  46 tasks      | elapsed:   10.8s
[Parallel(n_jobs=-1)]: Done 196 tasks      | elapsed:   44.4s
[Parallel(n_jobs=-1)]: Done 200 out of 200 | elapsed:   45.5s finished
[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  46 tasks      | elapsed:    0.0s
[Parallel(n_jobs=2)]: Done 196 tasks      | elapsed:    0.1s
[Parallel(n_jobs=2)]: Done 200 out of 200 | elapsed:    0.1s finished


Predicting...
Fold Macro F1: 0.6999
Fold duration: 45.73 sec

Fold 3/5
----------------------------------------
Train samples: 7957
Test samples : 1989
Training model...


...

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=-1)]: Done  46 tasks      | elapsed:   11.2s
[Parallel(n_jobs=-1)]: Done 196 tasks      | elapsed:   46.2s
[Parallel(n_jobs=-1)]: Done 200 out of 200 | elapsed:   46.9s finished
[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  46 tasks      | elapsed:    0.0s
[Parallel(n_jobs=2)]: Done 196 tasks      | elapsed:    0.1s
[Parallel(n_jobs=2)]: Done 200 out of 200 | elapsed:    0.1s finished
[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 2 concurrent workers.


Predicting...
Fold Macro F1: 0.6873
Fold duration: 47.13 sec

Fold 5/5
----------------------------------------
Train samples: 7957
Test samples : 1989
Training model...


[Parallel(n_jobs=-1)]: Done  46 tasks      | elapsed:   11.0s
[Parallel(n_jobs=-1)]: Done 196 tasks      | elapsed:   43.7s


Predicting...
Fold Macro F1: 0.6516
Fold duration: 44.70 sec

Cross-validation completed.
Total CV time: 3.81 minutes



[Parallel(n_jobs=-1)]: Done 200 out of 200 | elapsed:   44.5s finished
[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  46 tasks      | elapsed:    0.0s
[Parallel(n_jobs=2)]: Done 196 tasks      | elapsed:    0.1s
[Parallel(n_jobs=2)]: Done 200 out of 200 | elapsed:    0.1s finished


## Cell 14: Evaluation Metrics

In [14]:
############################################
# METRICS
############################################

print("=====================================================")
print(" EVALUATION")
print("=====================================================")
print()

cm = confusion_matrix(y, y_pred)

report = classification_report(
    y,
    y_pred,
    output_dict=True
)

macro_f1 = f1_score(
    y,
    y_pred,
    average="macro"
)

balanced_acc = balanced_accuracy_score(
    y,
    y_pred
)

print(f"Macro F1 Score     : {macro_f1:.4f}")
print(f"Balanced Accuracy  : {balanced_acc:.4f}\n")

 EVALUATION

Macro F1 Score     : 0.7003
Balanced Accuracy  : 0.6954



## Cell 15: Final Model Training

In [15]:
############################################
# TRAIN FINAL MODEL
############################################

print("=====================================================")
print(" FINAL TRAINING")
print("=====================================================")
print()

final_train_start = time.time()

rf.fit(
    X,
    y,
    sample_weight=weights
)

final_train_elapsed = time.time() - final_train_start

print(f"Final model trained in {final_train_elapsed/60:.2f} minutes\n")

 FINAL TRAINING



[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=-1)]: Done  46 tasks      | elapsed:   13.7s
[Parallel(n_jobs=-1)]: Done 196 tasks      | elapsed:   58.3s


Final model trained in 0.99 minutes



[Parallel(n_jobs=-1)]: Done 200 out of 200 | elapsed:   59.3s finished


## Cell 16: Save Results

In [16]:
############################################
# SAVE
############################################

print("Saving outputs...")

model_path = os.path.join(
    path_rf_output,
    f"RF_{next_x}_model.pkl"
)

joblib.dump(rf, model_path)

# Save PCA if used
if use_pca_reduction:
    pca_path = os.path.join(
        path_rf_output,
        f"RF_{next_x}_pca.pkl"
    )
    joblib.dump(pca, pca_path)
    print(f"PCA saved to        : {pca_path}")

cm_path = os.path.join(
    path_rf_output,
    f"confusion_matrix_RF_{next_x}.csv"
)

pd.DataFrame(cm).to_csv(
    cm_path,
    index=False
)

print(f"Model saved to      : {model_path}")
print(f"Confusion matrix    : {cm_path}")

Saving outputs...
Model saved to      : /content/drive/MyDrive/audio_simon_moutier/Results/Random_Forest/RF_3/RF_3_model.pkl
Confusion matrix    : /content/drive/MyDrive/audio_simon_moutier/Results/Random_Forest/RF_3/confusion_matrix_RF_3.csv


## Cell 17: Generate Summary Report

In [ ]:
############################################
# REPORT
############################################

summary_path = os.path.join(
    path_rf_output,
    f"summary_RF_{next_x}.txt"
)

with open(summary_path, "w") as f:

    f.write("====================================================\n")
    f.write(" RANDOM FOREST SUMMARY\n")
    f.write("====================================================\n\n")

    f.write(f"Date: {datetime.now()}\n\n")

    f.write(f"Macro F1 Score    : {macro_f1:.4f}\n")
    f.write(f"Balanced Accuracy : {balanced_acc:.4f}\n\n")

    f.write(f"Total samples     : {len(df)}\n")
    f.write(f"Number of features: {len(feature_cols)}\n")
    if use_pca_reduction:
        f.write(f"Features after PCA: {n_pca_components}\n")
        f.write(f"Explained variance: {pca.explained_variance_ratio_.sum():.4f}\n")
    f.write(f"Number of classes : {y.nunique()}\n\n")

    f.write("Model configuration:\n")
    f.write(f"  - n_estimators: 200\n")
    f.write(f"  - min_samples_leaf: 5\n")
    f.write(f"  - CV splits: {cv_splits}\n\n")

    f.write("Class distribution:\n")
    f.write(str(y.value_counts()))
    f.write("\n\n")

    f.write("Classification report:\n")
    f.write(str(classification_report(y, y_pred)))

total_elapsed = time.time() - start_time

print("\n=====================================================")
print(" TRAINING COMPLETED")
print("=====================================================")
print(f"Total runtime: {total_elapsed/60:.2f} minutes")
print(f"Results saved in:\n{path_rf_output}")
print("=====================================================")


 TRAINING COMPLETED
Total runtime: 13.86 minutes
Results saved in:
/content/drive/MyDrive/audio_simon_moutier/Results/Random_Forest/RF_3
